# Creating a Simple Agent with Tracing

In [15]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()


True

In [16]:

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)


We are up and running!


In [17]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [18]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [19]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Generally very healthy when eaten as part of a balanced diet.
    
    Key points:
    - Nutrients: potassium (~400 mg), vitamin B6, vitamin C, dietary fiber.
    - Low fat; natural sugars but in a wholesome package.
    - Calorie roughly 90–105 per medium banana.
    - Benefits: heart health, digestion, steady energy.
    - Considerations: ripe bananas have more sugar; diabetics or those watching sugar intake may want portions; green bananas have more resistant starch (lower GI).
    - Quick tip: pair with protein or fat (e.g., peanut butter) for satiety.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [20]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas with three bullet points?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

- Nutrient-dense: good source of potassium, vitamin B6, vitamin C and dietary fiber.  
- Supports heart health and digestion (potassium helps blood pressure; fiber aids gut health).  
- Moderation matters: ripe bananas have more sugar; great as a quick, portable snack when balanced with other foods.

_Good Job!_